# Pytorch Basics
## Back propagation

Automatic Differentiation with `torch.autograd`
===============================================


When training neural networks, the most frequently used algorithm is
**back propagation**. In this algorithm, parameters (model weights) are
adjusted according to the **gradient** of the loss function with respect
to the given parameter.

To compute those gradients, PyTorch has a built-in differentiation
engine called `torch.autograd`. It supports automatic computation of
gradient for any computational graph.

Consider the simplest one-layer neural network, with input `x`,
parameters `w` and `b`, and some loss function. It can be defined in
PyTorch in the following manner:

![](https://pytorch.org/tutorials/_static/img/basics/comp-graph.png)

In this network, `w` and `b` are **parameters**, which we need to
optimize. Thus, we need to be able to compute the gradients of loss
function with respect to those variables. In order to do that, we set
the `requires_grad` property of those tensors.


### 1. Import torch and other required libraries

```import torch ```

In [1]:
import torch

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


### 2. Declare the nessesary tensors following the computer graph 

For `x` ,a.k.a the input declare a tensor of ones with 5 elements 

For `w` , a.ka. the weigths, declare a matrix with 5 rows and 3 colums , check ```randn``` , and use the property ```requires_grad=True```. we need to keep track of the changes

For `b` , a.k.a bias , declare a tensor of 3 random numbers , also use the property ```requires_grad=True```

Create the value for `z` by using the matrix multiplication between `x` and `w` and add the values of `b`


For `y` declare a tensor of "expected output" 

`CE` is the funtional that will compute the difference between the prediction and the expected output (or the real values in the data). Use the `binary_cross_entropy_with_logits` funtional 


Get the value of the `loss` (the differece) between the predicted values and the expected values 



<details><summary><b>Solution</b></summary>
<pre>


```python
x = torch.ones(5)  # input tensor

w = torch.randn(5, 3, requires_grad=True)

b = torch.randn(3, requires_grad=True)

z = torch.matmul(x, w)+b


y = torch.zeros(3)  # expected output

loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)
```
</pre>
</details>

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>You can set the value of <code>requires_grad</code> when creating atensor, or later by using 
    <code>x.requires_grad_(True)</code> method.</p>

</div>


A function that we apply to tensors to construct computational graph is
in fact an object of class `Function`. This object knows how to compute
the function in the *forward* direction, and also how to compute its
derivative during the *backward propagation* step. A reference to the
backward propagation function is stored in `grad_fn` property of a
tensor. You can find more information of `Function` [in the
documentation](https://pytorch.org/docs/stable/autograd.html#function).

Check the `grad_fn` property of the tensors `z` and `loss`


<details><summary><b>Solution</b></summary>
<pre>

```python
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")
```

</pre>
</details>

Computing Gradients
===================

To optimize weights of parameters in the neural network, we need to
compute the derivatives of our loss function with respect to parameters,
namely, we need $\frac{\partial loss}{\partial w}$ and
$\frac{\partial loss}{\partial b}$ under some fixed values of `x` and
`y`. To compute those derivatives, we call `loss.backward()`, and then
retrieve the values from `w.grad` and `b.grad`:

```loss.backward()```

```print(w.grad)```

```print(b.grad)```

When we call `loss.backward()`, the whole graph is differentiated w.r.t.
the loss, and all Tensors in the graph that has `requires_grad=True` will
have their `.grad` Tensor accumulated with the gradient.

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<ul>
<li>We can only obtain the <code>grad</code> properties for the leafnodes of the computational graph, which have <code>requires_grad</code> property set to <code>True</code>. For all other nodes in our graph, gradients will not be available.- We can only perform gradient calculations using<code>backward</code> once on a given graph, for performance reasons. If we needto do several <code>backward</code> calls on the same graph, we need to pass<code>retain_graph=True</code> to the <code>backward</code> call.</li>
</ul>

</div>


Disabling Gradient Tracking
===========================

By default, all tensors with `requires_grad=True` are tracking their
computational history and support gradient computation. However, there
are some cases when we do not need to do that, for example, when we have
trained the model and just want to apply it to some input data, i.e. we
only want to do *forward* computations through the network. We can stop
tracking computations by surrounding our computation code with
`torch.no_grad()` block:


Again calculate the value of `z` and `loss` , but this time use the `torch.no_grad()` block to stop tracking the computations. Print the value of `z.requires_grad` and `loss.requires_grad`

<details><summary><b>Solution</b></summary>
<pre>

```python
z = torch.matmul(x, w)+b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

```

</pre>
</details>

Another way to achieve the same result is to use the `detach()` method
on the tensor, try this method on the tensor `z` and print the value of `z.requires_grad`:

<details><summary><b>Solution</b></summary>
<pre>

```python
z = torch.matmul(x, w)+b
z_det = z.detach()
print(z_det.requires_grad)
```

</pre>
</details>

There are reasons you might want to disable gradient tracking:

 -   To mark some parameters in your neural network as **frozen
        parameters**.
   
 -   To **speed up computations** when you are only doing forward
        pass, because computations on tensors that do not track
        gradients would be more efficient.


More on Computational Graphs
============================

Conceptually, autograd keeps a record of data (tensors) and all executed
operations (along with the resulting new tensors) in a directed acyclic
graph (DAG) consisting of
[Function](https://pytorch.org/docs/stable/autograd.html#torch.autograd.Function)
objects. In this DAG, leaves are the input tensors, roots are the output
tensors. By tracing this graph from roots to leaves, you can
automatically compute the gradients using the chain rule.

In a forward pass, autograd does two things simultaneously:

-   run the requested operation to compute a resulting tensor
-   maintain the operation's *gradient function* in the DAG.

The backward pass kicks off when `.backward()` is called on the DAG
root. `autograd` then:

-   computes the gradients from each `.grad_fn`,
-   accumulates them in the respective tensor's `.grad` attribute
-   using the chain rule, propagates all the way to the leaf tensors.

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>An important thing to note is that the graph is recreated from scratch; after each<code>.backward()</code> call, autograd starts populating a new graph. This isexactly what allows you to use control flow statements in your model;you can change the shape, size and operations at every iteration ifneeded.</p>

</div>


Tensor Gradients 
========================================================
In many cases, we have a scalar loss function, and we need to compute
the gradient with respect to some parameters. However, there are cases
when the output function is an arbitrary tensor. In this case, PyTorch
allows you to compute so-called **Jacobian product**, and not the actual
gradient.

For a vector function $\vec{y}=f(\vec{x})$, where
$\vec{x}=\langle x_1,\dots,x_n\rangle$ and
$\vec{y}=\langle y_1,\dots,y_m\rangle$, a gradient of $\vec{y}$ with
respect to $\vec{x}$ is given by **Jacobian matrix**:

$$\begin{aligned}
J=\left(\begin{array}{ccc}
   \frac{\partial y_{1}}{\partial x_{1}} & \cdots & \frac{\partial y_{1}}{\partial x_{n}}\\
   \vdots & \ddots & \vdots\\
   \frac{\partial y_{m}}{\partial x_{1}} & \cdots & \frac{\partial y_{m}}{\partial x_{n}}
   \end{array}\right)
\end{aligned}$$

Instead of computing the Jacobian matrix itself, PyTorch allows you to
compute **Jacobian Product** $v^T\cdot J$ for a given input vector
$v=(v_1 \dots v_m)$. This is achieved by calling `backward` with $v$ as
an argument. The size of $v$ should be the same as the size of the
original tensor, with respect to which we want to compute the product:


Lets start with a example , Create a 4x5 identity matrix with requires_grad=True to track computations. Use `torch.eye` to create the matrix, and make sure to set the `requires_grad=True` property.

<details><summary><b>Solution</b></summary>
<pre>
```python
inp = torch.eye(4, 5, requires_grad=True)
```

</pre>
</details>

Using the previous matrix `inp` , create a tensor `out` by multiplying the matrix `inp` by 3.

<details><summary><b>Solution</b></summary>
<pre>

```python
out = (inp + 1).pow(2).t()
```

</pre>
</details>

Using the previous tensor `out` , create a tensor `out.backward()` with the value of the matrix `torch.ones_like(out)` as an argument (this means use the tensor inside the `backward()` ). Remember to make sure to set the `requires_grad=True` property. Print the value of the `inp.grad` tensor like this `print(f"First call\n{inp.grad}")`


First call
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])


<details><summary><b>Solution</b></summary>
<pre>

```python

out.backward(torch.tensor([1, 0.1, 0.01, 0.001]), retain_graph=True)
print(f"First call\n{inp.grad}")

```

</pre>
</details>

Lets do a second call to the `out.backward()` method, this time use the tensor `torch.ones_like(out)` as an argument. Print the value of the `inp.grad` tensor like this `print(f"Second call\n{inp.grad}")`

In [11]:


# Perform backpropagation again with the same tensor of ones, retain the graph
out.backward(torch.ones_like(out), retain_graph=True)
# Print the gradients of inp after the second backward call
print(f"\nSecond call\n{inp.grad}")



Second call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])


<details><summary><b>Solution</b></summary>
<pre>

```python
out.backward(torch.ones_like(out), retain_graph=True)

print(f"\nSecond call\n{inp.grad}")
```

</pre>
</details>

Now lets return the gradient to zero by using the `zero_()` method on the tensor `inp.grad` and print the value of the `inp.grad` tensor like this `print(f"Zero the gradient\n{inp.grad}")`

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

<details><summary><b>Solution</b></summary>
<pre>

```python
inp.grad.zero_()
print(f"Zero the gradient\n{inp.grad}")
```

</pre>
</details>

Now lets do a third call to the `out.backward()` method, this time use the tensor `torch.ones_like(out)` as an argument. Print the value of the `inp.grad` tensor like this `print(f"Call after zeroing gradients\n{inp.grad}")`


Call after zeroing gradients
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])


<details><summary><b>Solution</b></summary>
<pre>

```python
out.backward(torch.ones_like(out), retain_graph=True)

print(f"\nCall after zeroing gradients\n{inp.grad}")
```

</pre>
</details>

Notice that when we call `backward` for the second time with the same
argument, the value of the gradient is different. This happens because
when doing `backward` propagation, PyTorch **accumulates the
gradients**, i.e. the value of computed gradients is added to the `grad`
property of all leaf nodes of computational graph. If you want to
compute the proper gradients, you need to zero out the `grad` property
before. In real-life training an *optimizer* helps us to do this.
